In [1]:
# ==========================================================
# BLOCK 1: BINARY SETUP & DATA LOADING
# ==========================================================
import warnings
warnings.filterwarnings('ignore')

import os
import random
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# 1. Global Reproducibility (Q1 Journal Requirement)
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 2. GPU Setup (Safe Memory Growth)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPUs Detected: {len(gpus)}")
    except RuntimeError as e:
        print(e)

# 3. Directories & Constants
DATASET_ROOT = '/home/T2430514/Downloads/MargeDataset/Binary'
ANOMALY_DIR = os.path.join(DATASET_ROOT, 'Anomaly')
NORMAL_DIR = os.path.join(DATASET_ROOT, 'Normal')
PROCESSED_DATA_DIR = '/home/T2430514/Downloads/MargeDataset/Processed' 
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

FRAME_SIZE = (224, 224)
NUM_FRAMES = 16
BATCH_SIZE = 4 

# 4. Data Loading Logic
def create_dataframe():
    data = []
    # Anomaly (Label 1)
    if os.path.exists(ANOMALY_DIR):
        for video_file in os.listdir(ANOMALY_DIR):
            if video_file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                data.append({'path': os.path.join(ANOMALY_DIR, video_file), 'bin_label': 1})
    
    # Normal (Label 0)
    if os.path.exists(NORMAL_DIR):
        for video_file in os.listdir(NORMAL_DIR):
            if video_file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                data.append({'path': os.path.join(NORMAL_DIR, video_file), 'bin_label': 0})
                
    return pd.DataFrame(data)

all_df = create_dataframe()
print(f"Total Binary Videos: {len(all_df)}")

# 5. Stratified Split (Crucial for Imbalanced/Small Data)
train_df, temp_df = train_test_split(all_df, test_size=0.2, stratify=all_df['bin_label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['bin_label'], random_state=SEED)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# 6. Compute Class Weights
y_train = train_df['bin_label'].values
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))
print(f"Class Weights: {class_weights_dict}")

2026-04-28 14:42:48.191774: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-28 14:42:48.230942: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777365768.241644 3485186 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777365768.245751 3485186 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777365768.280780 3485186 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

GPUs Detected: 1
Total Binary Videos: 4534
Train: 3627 | Val: 453 | Test: 454
Class Weights: {0: np.float64(1.0002757859900717), 1: np.float64(0.9997243660418964)}


In [2]:
# ==========================================================
# BLOCK 2: FRAME EXTRACTION
# ==========================================================
import sys

def extract_and_save_frames(dataframe, output_dir, num_frames=NUM_FRAMES, frame_size=FRAME_SIZE):
    print(f"Processing {len(dataframe)} videos...")
    count = 0
    
    for idx, row in dataframe.iterrows():
        base_name = os.path.basename(row.path)
        # Use .npy for faster loading during training
        save_name = os.path.splitext(base_name)[0] + '.npy'
        output_path = os.path.join(output_dir, save_name)
        
        # Skip if already processed
        if os.path.exists(output_path): 
            continue

        cap = cv2.VideoCapture(row.path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if total_frames <= 0:
            cap.release()
            continue
            
        # Uniform Temporal Sampling (SOTA standard)
        frame_indices = np.linspace(0, max(total_frames - 1, 0), num=num_frames, dtype=int)
        frames = []
        
        for i in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, frame_size)
                frames.append(frame)
            else:
                # Padding if read fails
                frames.append(np.zeros(frame_size + (3,), dtype=np.uint8))
        cap.release()
        
        # Ensure exact frame count
        while len(frames) < num_frames:
            frames.append(np.zeros(frame_size + (3,), dtype=np.uint8))
            
        # Save as uint8 to save disk space (converted to float32 in generator)
        np.save(output_path, np.array(frames, dtype=np.uint8))
        
        count += 1
        if count % 100 == 0: 
            sys.stdout.write(f"\rExtracted {count} videos.")
    print("\nExtraction Complete.")

extract_and_save_frames(all_df, PROCESSED_DATA_DIR)

Processing 4534 videos...

Extraction Complete.


In [3]:
# ==========================================================
# BLOCK 3: SOTA DATA GENERATOR (EXPERIMENT C: CUTMIX)
# ==========================================================
import tensorflow as tf
import numpy as np
import os

class CutMixVideoDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, processed_data_dir, batch_size=BATCH_SIZE, 
                 num_frames=NUM_FRAMES, frame_size=FRAME_SIZE, 
                 augment=False, shuffle=True, cutmix_alpha=1.0):
        self.dataframe = dataframe
        self.processed_data_dir = processed_data_dir
        self.batch_size = batch_size
        self.num_frames = num_frames
        self.frame_size = frame_size
        self.augment = augment
        self.shuffle = shuffle
        self.cutmix_alpha = cutmix_alpha
        self.indices = np.arange(len(self.dataframe))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.dataframe) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        return self.__data_generation(batch_indices)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def load_video(self, video_path):
        base_name = os.path.basename(video_path)
        name, _ = os.path.splitext(base_name)
        npy_path = os.path.join(self.processed_data_dir, name + '.npy')
        
        if os.path.exists(npy_path):
            try: 
                return np.load(npy_path).astype(np.float32) / 255.0
            except: 
                pass
        return np.zeros((self.num_frames, *self.frame_size, 3), dtype=np.float32)

    def rand_bbox(self, size, lam):
        H, W = size
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)

        cx = np.random.randint(W)
        cy = np.random.randint(H)

        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)

        return bbx1, bby1, bbx2, bby2

    def apply_consistent_augmentation(self, video):
        """
        Applies Memory-Safe Augmentations using pure NumPy.
        Prevents TensorFlow EagerTensor RAM leaks.
        """
        # 1. Random Horizontal Flip (axis=2 is the width dimension)
        if np.random.rand() > 0.5:
            video = np.flip(video, axis=2)
            
        # 2. Random Brightness (-0.15 to 0.15)
        brightness_delta = np.random.uniform(-0.15, 0.15)
        video = video + brightness_delta
        
        # 3. Random Contrast (0.85 to 1.15)
        contrast_factor = np.random.uniform(0.85, 1.15)
        mean = np.mean(video, axis=(1, 2, 3), keepdims=True)
        video = (video - mean) * contrast_factor + mean
        
        # Clip strictly to [0.0, 1.0] and ensure float32
        return np.clip(video, 0.0, 1.0).astype(np.float32)

    def __data_generation(self, batch_indices):
        X = np.empty((self.batch_size, self.num_frames, *self.frame_size, 3), dtype=np.float32)
        y = np.empty((self.batch_size), dtype=np.float32)

        for i, idx in enumerate(batch_indices):
            row = self.dataframe.iloc[idx]
            video = self.load_video(row.path)
            label = float(row.bin_label)

            # --- SOTA: TUBE CUTMIX LOGIC ---
            if self.augment and self.cutmix_alpha > 0 and np.random.rand() < 0.5:
                rand_idx = np.random.choice(self.indices)
                row2 = self.dataframe.iloc[rand_idx]
                video2 = self.load_video(row2.path)
                label2 = float(row2.bin_label)

                lam = np.random.beta(self.cutmix_alpha, self.cutmix_alpha)
                bbx1, bby1, bbx2, bby2 = self.rand_bbox(self.frame_size, lam)
                
                # Tube CutMix (applies patch identically through all frames)
                video[:, bby1:bby2, bbx1:bbx2, :] = video2[:, bby1:bby2, bbx1:bbx2, :]
                
                # Adjust label based on patch area
                patch_ratio = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (self.frame_size[0] * self.frame_size[1]))
                label = patch_ratio * label + (1 - patch_ratio) * label2

            # --- SPATIAL AUGMENTATION ---
            if self.augment:
                video = self.apply_consistent_augmentation(video)

            X[i,] = video
            y[i] = label

        return X, y
    
    def get_labels(self):
        original_indices = self.indices.copy()
        if self.shuffle:
            sorted_indices = np.arange(len(self.dataframe))
        else:
            sorted_indices = self.indices
            
        limit = self.__len__() * self.batch_size
        return self.dataframe.iloc[sorted_indices[:limit]]['bin_label'].values

# Alias for model blocks to use seamlessly
SOTAVideoDataGenerator = CutMixVideoDataGenerator

print("Initializing Generators (Experiment C: CutMix Augmentation - NUMPY FIX)...")
train_generator = SOTAVideoDataGenerator(
    train_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=True, cutmix_alpha=1.0, shuffle=True
)
val_generator = SOTAVideoDataGenerator(
    val_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False
)
print("CutMix Generators Ready.")

Initializing Generators (Experiment C: CutMix Augmentation - NUMPY FIX)...
CutMix Generators Ready.


In [4]:
# ==========================================================
# BLOCK 4: NANO3D MODEL TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Nano3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Nano3D Architecture
# ----------------------------------------------------------
def se_block_3d(x, filters, squeeze_ratio=0.25):
    """Squeeze-and-Excitation for attention-driven channel weighting."""
    squeeze_channels = max(1, int(filters * squeeze_ratio))
    se = layers.GlobalAveragePooling3D()(x)
    se = layers.Reshape((1, 1, 1, filters))(se)
    se = layers.Dense(squeeze_channels, activation=tf.nn.swish, use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    return layers.Multiply()([x, se])

def nano_block(x, filters, strides=(1,1,1)):
    """Factorized Spatiotemporal Block with SE & Residual Connection."""
    shortcut = x
    
    # Spatial feature extraction
    x = layers.Conv3D(filters, (1,3,3), strides=(1, strides[1], strides[2]), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Temporal feature extraction
    x = layers.Conv3D(filters, (3,1,1), strides=(strides[0], 1, 1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Channel Attention
    x = se_block_3d(x, filters)
    
    # Residual matching
    if strides != (1,1,1) or shortcut.shape[-1] != filters:
        shortcut = layers.Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
        
    x = layers.Add()([x, shortcut])
    return x

def create_nano3d_model(input_shape):
    video_input = layers.Input(shape=input_shape)
    
    # --- Stem ---
    x = layers.Conv3D(16, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(video_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.Conv3D(16, (3,1,1), strides=(1,1,1), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.MaxPooling3D((1,2,2), strides=(1,2,2), padding='same')(x)
    
    # --- Hierarchical Blocks ---
    # Block 1 (Low-level features)
    b1 = nano_block(x, 32)
    b1 = nano_block(b1, 32)
    
    # Block 2 (Mid-level features)
    b2 = nano_block(b1, 64, strides=(1,2,2))
    b2 = nano_block(b2, 64)
    
    # Block 3 (High-level features)
    b3 = nano_block(b2, 128, strides=(2,2,2))
    b3 = nano_block(b3, 128)
    
    # --- Multi-Scale Feature Fusion (DenseNet replacement) ---
    pool1 = layers.GlobalAveragePooling3D()(b1)
    pool2 = layers.GlobalAveragePooling3D()(b2)
    pool3 = layers.GlobalAveragePooling3D()(b3)
    
    merged_features = layers.Concatenate()([pool1, pool2, pool3])
    
    # --- Classification Head ---
    x = layers.Dropout(0.4)(merged_features)
    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name='Nano3D')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_nano3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_nano3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Nano3D.


I0000 00:00:1777365782.462206 3485186 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13582 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9



Training Model: Nano3D...
Epoch 1/50


I0000 00:00:1777365787.015616 3485289 service.cc:152] XLA service 0x70d4280027b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777365787.015637 3485289 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-04-28 14:43:07.204370: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777365788.062803 3485289 cuda_dnn.cc:529] Loaded cuDNN version 91002


  4/906 ━━━━━━━━━━━━━━━━━━━━ 41s 46ms/step - accuracy: 0.4688 - auc: 0.5310 - loss: 0.7373 

I0000 00:00:1777365795.213513 3485289 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


906/906 ━━━━━━━━━━━━━━━━━━━━ 64s 57ms/step - accuracy: 0.5428 - auc: 0.7472 - loss: 0.6425 - val_accuracy: 0.7832 - val_auc: 0.8685 - val_loss: 0.5382
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 46ms/step - accuracy: 0.5673 - auc: 0.7791 - loss: 0.6134 - val_accuracy: 0.7920 - val_auc: 0.8839 - val_loss: 0.5290
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 46ms/step - accuracy: 0.5831 - auc: 0.8019 - loss: 0.5913 - val_accuracy: 0.8142 - val_auc: 0.8846 - val_loss: 0.5202
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 46ms/step - accuracy: 0.6007 - auc: 0.8086 - loss: 0.5788 - val_accuracy: 0.7965 - val_auc: 0.8804 - val_loss: 0.5616
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 43s 47ms/step - accuracy: 0.6046 - auc: 0.8186 - loss: 0.5732 - val_accuracy: 0.8142 - val_auc: 0.8900 - val_loss: 0.5504
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 46ms/step - accuracy: 0.6217 - auc: 0.8226 - loss: 0.5589 - val_accuracy: 0.8119 - val_auc: 0.9010 - val_loss: 0.5045
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [5]:
# ==========================================================
# BLOCK 40 (XAI 1): BATCH 3D GRAD-CAM GENERATOR FOR NANO3D
# ==========================================================
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from scipy.ndimage import zoom

# 1. Setup Output Directory
# ----------------------------------------------------------
XAI_OUT_DIR = "thesis_xai_outputs"
os.makedirs(XAI_OUT_DIR, exist_ok=True)
print(f"XAI outputs will be saved to: {XAI_OUT_DIR}/")

# 2. Load Model Safely (Handling Custom Swish)
# ----------------------------------------------------------
print("Loading Nano3D Model...")
try:
    model = load_model('best_nano3d_model.keras', custom_objects={'swish': tf.nn.swish})
except Exception as e:
    print("Error loading model. Ensure you have run the Nano3D training block.")
    raise e

# Dynamically find the last 3D Convolutional Layer for Grad-CAM
last_conv_layer_name = None
for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Conv3D):
        last_conv_layer_name = layer.name
        break

print(f"Targeting Layer for Grad-CAM: {last_conv_layer_name}")

grad_model = tf.keras.models.Model(
    [model.inputs], 
    [model.get_layer(last_conv_layer_name).output, model.output]
)

# 3. Helper: Generate 3D Grad-CAM
# ----------------------------------------------------------
def compute_3d_gradcam(video_tensor):
    """Computes the 3D Gradient-weighted Class Activation Mapping."""
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(video_tensor)
        # Nano3D uses a sigmoid output (1 node). We want the gradient of the anomaly score.
        loss = predictions[:, 0]

    # Extract gradients and pool them spatially and temporally
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2, 3))

    # Weight the convolutional feature maps by the pooled gradients
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)
    
    # Apply ReLU to keep only features that positively contribute to the "Anomaly" class
    heatmap = np.maximum(heatmap, 0)
    
    # Normalize heatmap to [0, 1]
    max_val = np.max(heatmap)
    if max_val == 0:
        return heatmap # Avoid division by zero if no activations
    heatmap /= max_val
    return heatmap

# 4. Helper: Find Top Anomaly Videos
# ----------------------------------------------------------
print("Scanning Test Dataset for the Best Anomaly Examples...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)
y_true = test_generator.get_labels()
y_pred_probs = model.predict(test_generator, verbose=1).flatten()

# Find indices where the video is actually an anomaly (1) AND the model was highly confident
anomaly_indices = np.where((y_true == 1) & (y_pred_probs > 0.85))[0]

# Sort them by highest confidence first
sorted_anomaly_indices = anomaly_indices[np.argsort(y_pred_probs[anomaly_indices])[::-1]]

# Take the top 10 best videos to generate images for
top_n_indices = sorted_anomaly_indices[:10]
print(f"Found {len(top_n_indices)} highly confident anomaly videos for batch generation.")

# 5. Batch Generation Loop
# ----------------------------------------------------------
for rank, idx in enumerate(top_n_indices):
    row = test_df.iloc[idx]
    confidence = y_pred_probs[idx]
    
    # Load the raw video array
    video_array = test_generator.load_video(row.path)
    video_tensor = tf.expand_dims(video_array, axis=0) # Add batch dimension
    
    # Compute the 3D low-res heatmap
    raw_heatmap_3d = compute_3d_gradcam(video_tensor)
    
    # Resize the heatmap to match the original video dimensions using scipy
    # Zoom factors: (Time_ratio, Height_ratio, Width_ratio)
    zoom_factors = (
        video_array.shape[0] / raw_heatmap_3d.shape[0],
        video_array.shape[1] / raw_heatmap_3d.shape[1],
        video_array.shape[2] / raw_heatmap_3d.shape[2]
    )
    resized_heatmap_3d = zoom(raw_heatmap_3d, zoom_factors, order=1) # order=1 is bilinear
    
    # Select 4 key frames evenly spaced across the video to display
    key_frames = np.linspace(0, video_array.shape[0] - 1, 4, dtype=int)
    
    # Create a Q1-Ready Matplotlib Figure (2 Rows: Original vs. Overlay)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8), dpi=300)
    fig.suptitle(f"Spatiotemporal Grad-CAM | Nano3D (CutMix) | Anomaly Confidence: {confidence:.4f}", fontsize=18, y=0.98)
    
    for col_idx, frame_idx in enumerate(key_frames):
        # Original Frame (RGB)
        frame_rgb = video_array[frame_idx]
        axes[0, col_idx].imshow(frame_rgb)
        axes[0, col_idx].set_title(f"Frame {frame_idx + 1}")
        axes[0, col_idx].axis('off')
        
        # Heatmap Overlay
        heatmap_frame = resized_heatmap_3d[frame_idx]
        # Convert heatmap to JET colormap (requires uint8)
        heatmap_img = np.uint8(255 * heatmap_frame)
        heatmap_color = cv2.applyColorMap(heatmap_img, cv2.COLORMAP_JET)
        heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB) # OpenCV uses BGR natively
        
        # Superimpose the heatmap onto the original frame (Alpha blending: 60% image, 40% heatmap)
        frame_uint8 = np.uint8(255 * frame_rgb)
        superimposed_img = cv2.addWeighted(frame_uint8, 0.6, heatmap_color, 0.4, 0)
        
        axes[1, col_idx].imshow(superimposed_img)
        axes[1, col_idx].set_title("Grad-CAM Activation")
        axes[1, col_idx].axis('off')

    plt.tight_layout()
    
    # Save image with zero white borders
    save_path = os.path.join(XAI_OUT_DIR, f"nano3d_anomaly_rank_{rank+1}_conf_{confidence:.2f}.png")
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close(fig) # Free up RAM

print(f"\n✅ Batch XAI Generation Complete! Check the '{XAI_OUT_DIR}/' folder to pick your favorite images.")

XAI outputs will be saved to: thesis_xai_outputs/
Loading Nano3D Model...
Targeting Layer for Grad-CAM: conv3d_16
Scanning Test Dataset for the Best Anomaly Examples...
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step
Found 10 highly confident anomaly videos for batch generation.

✅ Batch XAI Generation Complete! Check the 'thesis_xai_outputs/' folder to pick your favorite images.


In [7]:
# ==========================================================
# BLOCK 41 (XAI 2): 3D SCORE-CAM GENERATOR FOR NANO3D (10 IMAGES)
# ==========================================================
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from scipy.ndimage import zoom

# 1. Setup Output Directory
# ----------------------------------------------------------
XAI_SCORECAM_DIR = "thesis_xai_scorecam"
os.makedirs(XAI_SCORECAM_DIR, exist_ok=True)
print(f"Score-CAM outputs will be saved to: {XAI_SCORECAM_DIR}/")

# 2. Load Model Safely 
# ----------------------------------------------------------
print("Loading Nano3D Model for Score-CAM...")
try:
    model = load_model('best_nano3d_model.keras', custom_objects={'swish': tf.nn.swish})
except Exception as e:
    print("Error loading model. Ensure Nano3D training is complete.")
    raise e

# Find the last 3D Convolutional Layer dynamically
last_conv_layer_name = None
for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Conv3D):
        last_conv_layer_name = layer.name
        break

print(f"Targeting Layer for Score-CAM: {last_conv_layer_name}")

feature_extractor = tf.keras.models.Model(
    inputs=model.inputs, 
    outputs=[model.get_layer(last_conv_layer_name).output, model.output]
)

# 3. Helper: Compute 3D Score-CAM (Memory-Safe Batching)
# ----------------------------------------------------------
def compute_scorecam_3d(video_tensor, batch_size=8):
    """Computes gradient-free Score-CAM for 3D Video."""
    # 1. Get raw feature maps
    conv_outputs, base_prediction = feature_extractor(video_tensor)
    conv_outputs = conv_outputs[0].numpy()  # Shape: (T_conv, H_conv, W_conv, Channels)
    num_channels = conv_outputs.shape[-1]
    
    input_shape = video_tensor.shape[1:4] # (T_in, H_in, W_in)
    
    # Zoom factors to upscale feature maps to original video resolution
    zoom_factors = (
        input_shape[0] / conv_outputs.shape[0],
        input_shape[1] / conv_outputs.shape[1],
        input_shape[2] / conv_outputs.shape[2]
    )
    
    masked_videos = []
    
    # 2. Upsample each channel and create masked videos
    for c in range(num_channels):
        activation_map = conv_outputs[..., c]
        
        # Upsample
        upsampled_map = zoom(activation_map, zoom_factors, order=1)
        
        # Normalize to [0, 1]
        map_min, map_max = np.min(upsampled_map), np.max(upsampled_map)
        if map_max - map_min > 1e-7:
            norm_map = (upsampled_map - map_min) / (map_max - map_min)
        else:
            norm_map = np.zeros_like(upsampled_map)
            
        # Broadcast mask to RGB channels and multiply with original video
        mask_rgb = np.expand_dims(norm_map, axis=-1)
        masked_video = video_tensor[0].numpy() * mask_rgb
        masked_videos.append(masked_video)
        
    masked_videos = np.array(masked_videos) # Shape: (Channels, T_in, H_in, W_in, 3)
    
    # 3. Forward pass masked videos in batches to prevent OOM
    channel_weights = []
    for i in range(0, num_channels, batch_size):
        batch = masked_videos[i : i + batch_size]
        scores = model.predict(batch, verbose=0).flatten()
        channel_weights.extend(scores)
        
    channel_weights = np.array(channel_weights)
    
    # 4. Weighted combination of original feature maps
    heatmap = np.zeros(conv_outputs.shape[0:3], dtype=np.float32)
    for c in range(num_channels):
        heatmap += channel_weights[c] * conv_outputs[..., c]
        
    # Apply ReLU and Normalize
    heatmap = np.maximum(heatmap, 0)
    max_val = np.max(heatmap)
    if max_val > 0:
        heatmap /= max_val
        
    return heatmap, base_prediction[0][0]

# 4. Helper: Find Top Anomaly Videos
# ----------------------------------------------------------
print("\nScanning Test Dataset for the Best Anomaly Examples...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)
y_true = test_generator.get_labels()
y_pred_probs = model.predict(test_generator, verbose=0).flatten()

anomaly_indices = np.where((y_true == 1) & (y_pred_probs > 0.85))[0]
sorted_anomaly_indices = anomaly_indices[np.argsort(y_pred_probs[anomaly_indices])[::-1]]

# Changed to 10 for batch generation
top_n_indices = sorted_anomaly_indices[:10]
print(f"Executing Score-CAM on Top {len(top_n_indices)} anomaly videos...\n")

# 5. Batch Generation Loop
# ----------------------------------------------------------
for rank, idx in enumerate(top_n_indices):
    row = test_df.iloc[idx]
    
    video_array = test_generator.load_video(row.path)
    video_tensor = tf.expand_dims(video_array, axis=0) 
    
    print(f"Processing Video Rank {rank+1} of {len(top_n_indices)}...")
    raw_heatmap_3d, confidence = compute_scorecam_3d(video_tensor, batch_size=8)
    
    # Resize the heatmap to match the original video dimensions
    zoom_factors = (
        video_array.shape[0] / raw_heatmap_3d.shape[0],
        video_array.shape[1] / raw_heatmap_3d.shape[1],
        video_array.shape[2] / raw_heatmap_3d.shape[2]
    )
    resized_heatmap_3d = zoom(raw_heatmap_3d, zoom_factors, order=1) 
    
    key_frames = np.linspace(0, video_array.shape[0] - 1, 4, dtype=int)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8), dpi=300)
    fig.suptitle(f"3D Score-CAM | Nano3D (CutMix) | Anomaly Confidence: {confidence:.4f}", fontsize=18, y=0.98)
    
    for col_idx, frame_idx in enumerate(key_frames):
        frame_rgb = video_array[frame_idx]
        axes[0, col_idx].imshow(frame_rgb)
        axes[0, col_idx].set_title(f"Frame {frame_idx + 1}")
        axes[0, col_idx].axis('off')
        
        heatmap_frame = resized_heatmap_3d[frame_idx]
        heatmap_img = np.uint8(255 * heatmap_frame)
        heatmap_color = cv2.applyColorMap(heatmap_img, cv2.COLORMAP_JET)
        heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB) 
        
        frame_uint8 = np.uint8(255 * frame_rgb)
        superimposed_img = cv2.addWeighted(frame_uint8, 0.6, heatmap_color, 0.4, 0)
        
        axes[1, col_idx].imshow(superimposed_img)
        axes[1, col_idx].set_title("Score-CAM Overlay")
        axes[1, col_idx].axis('off')

    plt.tight_layout()
    save_path = os.path.join(XAI_SCORECAM_DIR, f"scorecam_rank_{rank+1}_conf_{confidence:.2f}.png")
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close(fig) 

print(f"\n✅ Score-CAM Generation Complete! Images saved in '{XAI_SCORECAM_DIR}/'.")

Score-CAM outputs will be saved to: thesis_xai_scorecam/
Loading Nano3D Model for Score-CAM...
Targeting Layer for Score-CAM: conv3d_16

Scanning Test Dataset for the Best Anomaly Examples...
Executing Score-CAM on Top 10 anomaly videos...

Processing Video Rank 1 of 10...
Processing Video Rank 2 of 10...
Processing Video Rank 3 of 10...
Processing Video Rank 4 of 10...
Processing Video Rank 5 of 10...
Processing Video Rank 6 of 10...
Processing Video Rank 7 of 10...
Processing Video Rank 8 of 10...
Processing Video Rank 9 of 10...
Processing Video Rank 10 of 10...

✅ Score-CAM Generation Complete! Images saved in 'thesis_xai_scorecam/'.


In [9]:
# ==========================================================
# BLOCK 42 (XAI 3): CAUSAL PERTURBATION (ERASE-AND-TEST) - 10 IMAGES
# ==========================================================
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from scipy.ndimage import zoom

# 1. Setup Output Directory
# ----------------------------------------------------------
XAI_PERTURB_DIR = "thesis_xai_perturbation"
os.makedirs(XAI_PERTURB_DIR, exist_ok=True)
print(f"Causal Perturbation outputs will be saved to: {XAI_PERTURB_DIR}/")

# 2. Load Model Safely 
# ----------------------------------------------------------
print("Loading Nano3D Model for Causal Testing...")
try:
    model = load_model('best_nano3d_model.keras', custom_objects={'swish': tf.nn.swish})
except Exception as e:
    print("Error loading model. Ensure Nano3D training is complete.")
    raise e

# Setup fast Grad-CAM for mask generation
last_conv_layer_name = None
for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Conv3D):
        last_conv_layer_name = layer.name
        break

grad_model = tf.keras.models.Model(
    [model.inputs], 
    [model.get_layer(last_conv_layer_name).output, model.output]
)

def get_activation_mask(video_tensor):
    """Generates a fast 3D mask of the most attended regions."""
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(video_tensor)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2, 3))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)
    heatmap = np.maximum(heatmap, 0)
    max_val = np.max(heatmap)
    if max_val > 0:
        heatmap /= max_val
    return heatmap

# 3. Find Top Anomaly Videos
# ----------------------------------------------------------
print("\nScanning Test Dataset for the Best Anomaly Examples...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)
y_true = test_generator.get_labels()
y_pred_probs = model.predict(test_generator, verbose=0).flatten()

anomaly_indices = np.where((y_true == 1) & (y_pred_probs > 0.85))[0]
sorted_anomaly_indices = anomaly_indices[np.argsort(y_pred_probs[anomaly_indices])[::-1]]

# Changed to 10 for a wider selection of causal proofs
top_n_indices = sorted_anomaly_indices[:10] 
print(f"Running Causal Perturbation on Top {len(top_n_indices)} anomaly videos...\n")

# 4. Perturbation & Evaluation Loop
# ----------------------------------------------------------
for rank, idx in enumerate(top_n_indices):
    row = test_df.iloc[idx]
    
    # Load original video
    video_array = test_generator.load_video(row.path)
    video_tensor = tf.expand_dims(video_array, axis=0) 
    
    # 1. Score Original Video
    original_score = y_pred_probs[idx]
    
    # 2. Get Mask & Resize to Video Dimensions
    raw_heatmap = get_activation_mask(video_tensor)
    zoom_factors = (
        video_array.shape[0] / raw_heatmap.shape[0],
        video_array.shape[1] / raw_heatmap.shape[1],
        video_array.shape[2] / raw_heatmap.shape[2]
    )
    resized_heatmap = zoom(raw_heatmap, zoom_factors, order=1)
    
    # 3. Create the "Erase" Mask (Black out pixels where attention > 40%)
    threshold = 0.40
    binary_mask = (resized_heatmap > threshold).astype(np.float32)
    binary_mask_rgb = np.expand_dims(binary_mask, axis=-1)
    
    # 4. Apply Perturbation (Keep pixels where mask is 0, black out where mask is 1)
    perturbed_video_array = video_array * (1.0 - binary_mask_rgb)
    perturbed_video_tensor = tf.expand_dims(perturbed_video_array, axis=0)
    
    # 5. Score the Perturbed Video
    perturbed_score = model.predict(perturbed_video_tensor, verbose=0)[0][0]
    score_drop = original_score - perturbed_score
    
    print(f"Rank {rank+1:2d} | Original Score: {original_score:.4f} -> Perturbed Score: {perturbed_score:.4f} (Drop: -{score_drop:.4f})")
    
    # 6. Plotting the Causal Proof
    key_frames = np.linspace(0, video_array.shape[0] - 1, 4, dtype=int)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8), dpi=300)
    fig.suptitle(f"Causal Perturbation | Nano3D (CutMix)\nScore Drop: {original_score:.4f} ➔ {perturbed_score:.4f}", fontsize=18, y=1.02, color='darkred')
    
    for col_idx, frame_idx in enumerate(key_frames):
        # Row 1: Original Frames
        axes[0, col_idx].imshow(video_array[frame_idx])
        if col_idx == 0:
            axes[0, col_idx].set_ylabel(f"Original\nScore: {original_score:.2f}", fontsize=14, fontweight='bold')
        axes[0, col_idx].set_title(f"Frame {frame_idx + 1}")
        axes[0, col_idx].set_xticks([])
        axes[0, col_idx].set_yticks([])
        
        # Row 2: Perturbed (Censored) Frames
        axes[1, col_idx].imshow(perturbed_video_array[frame_idx])
        if col_idx == 0:
            axes[1, col_idx].set_ylabel(f"Perturbed\nScore: {perturbed_score:.2f}", fontsize=14, fontweight='bold')
        axes[1, col_idx].set_xticks([])
        axes[1, col_idx].set_yticks([])

    plt.tight_layout()
    save_path = os.path.join(XAI_PERTURB_DIR, f"causal_rank_{rank+1}_drop_{score_drop:.2f}.png")
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close(fig) 

print(f"\n✅ Causal Perturbation Complete! 10 Images saved in '{XAI_PERTURB_DIR}/'.")

Causal Perturbation outputs will be saved to: thesis_xai_perturbation/
Loading Nano3D Model for Causal Testing...

Scanning Test Dataset for the Best Anomaly Examples...
Running Causal Perturbation on Top 10 anomaly videos...

Rank  1 | Original Score: 0.9891 -> Perturbed Score: 0.9872 (Drop: -0.0019)
Rank  2 | Original Score: 0.9886 -> Perturbed Score: 0.9846 (Drop: -0.0040)
Rank  3 | Original Score: 0.9826 -> Perturbed Score: 0.9722 (Drop: -0.0104)
Rank  4 | Original Score: 0.9803 -> Perturbed Score: 0.9711 (Drop: -0.0091)
Rank  5 | Original Score: 0.9776 -> Perturbed Score: 0.9682 (Drop: -0.0095)
Rank  6 | Original Score: 0.9759 -> Perturbed Score: 0.9431 (Drop: -0.0328)
Rank  7 | Original Score: 0.9758 -> Perturbed Score: 0.9760 (Drop: --0.0002)
Rank  8 | Original Score: 0.9735 -> Perturbed Score: 0.9568 (Drop: -0.0167)
Rank  9 | Original Score: 0.9730 -> Perturbed Score: 0.9736 (Drop: --0.0006)
Rank 10 | Original Score: 0.9724 -> Perturbed Score: 0.9708 (Drop: -0.0016)

✅ Causal P

In [10]:
# ==========================================================
# BLOCK 43 (XAI 4): TEMPORAL CONFIDENCE TRAJECTORY
# ==========================================================
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# 1. Setup Output Directory
# ----------------------------------------------------------
XAI_TRAJECTORY_DIR = "thesis_xai_trajectory"
os.makedirs(XAI_TRAJECTORY_DIR, exist_ok=True)
print(f"Temporal Trajectory outputs will be saved to: {XAI_TRAJECTORY_DIR}/")

# 2. Load Model Safely 
# ----------------------------------------------------------
print("Loading Nano3D Model for Temporal Analysis...")
try:
    model = load_model('best_nano3d_model.keras', custom_objects={'swish': tf.nn.swish})
except Exception as e:
    print("Error loading model. Ensure Nano3D training is complete.")
    raise e

# 3. Find Top Anomaly Videos
# ----------------------------------------------------------
print("\nScanning Test Dataset for the Best Anomaly Examples...")
# Ensure generator matches your existing setup
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)
y_true = test_generator.get_labels()
y_pred_probs = model.predict(test_generator, verbose=0).flatten()

anomaly_indices = np.where((y_true == 1) & (y_pred_probs > 0.85))[0]
sorted_anomaly_indices = anomaly_indices[np.argsort(y_pred_probs[anomaly_indices])[::-1]]

# Target the top 10 examples for a wide selection
top_n_indices = sorted_anomaly_indices[:10] 
print(f"Executing Temporal Unrolling on Top {len(top_n_indices)} anomaly videos...\n")

# 4. Temporal Unrolling & Evaluation Loop
# ----------------------------------------------------------
for rank, idx in enumerate(top_n_indices):
    row = test_df.iloc[idx]
    
    # Load original video (Shape: [NUM_FRAMES, H, W, 3])
    video_array = test_generator.load_video(row.path)
    num_frames = video_array.shape[0]
    
    temporal_scores = []
    
    # Iteratively reveal frames one by one
    for t in range(1, num_frames + 1):
        # Create a masked video where future frames are blacked out (zeros)
        masked_video = np.copy(video_array)
        if t < num_frames:
            masked_video[t:, :, :, :] = 0.0
            
        video_tensor = tf.expand_dims(masked_video, axis=0)
        
        # Predict score for this time step
        step_score = model.predict(video_tensor, verbose=0)[0][0]
        temporal_scores.append(step_score)
        
    final_score = temporal_scores[-1]
    print(f"Rank {rank+1:2d} | Final Score: {final_score:.4f} | Max Jump: +{np.max(np.diff(temporal_scores)):.4f}")
    
    # 5. Plotting the Trajectory and Filmstrip
    fig = plt.figure(figsize=(14, 8), dpi=300)
    fig.suptitle(f"Temporal Confidence Trajectory | Nano3D (CutMix)\nFinal Anomaly Score: {final_score:.4f}", fontsize=18, y=0.98)
    
    # Top Plot: The Time-Series Line Chart
    ax1 = plt.subplot2grid((3, 4), (0, 0), colspan=4, rowspan=2)
    frames_x = np.arange(1, num_frames + 1)
    
    ax1.plot(frames_x, temporal_scores, marker='o', linestyle='-', color='darkred', linewidth=3, markersize=8)
    ax1.fill_between(frames_x, temporal_scores, alpha=0.2, color='red')
    
    ax1.set_xlim(1, num_frames)
    ax1.set_ylim(0.0, 1.05)
    ax1.set_xticks(frames_x)
    ax1.set_xlabel("Frames Revealed (Time)", fontsize=12, fontweight='bold')
    ax1.set_ylabel("Anomaly Confidence", fontsize=12, fontweight='bold')
    ax1.grid(True, linestyle='--', alpha=0.6)
    
    # Highlight the frame with the highest confidence jump (The "Trigger" moment)
    score_diffs = np.diff(temporal_scores)
    trigger_frame_idx = np.argmax(score_diffs) + 1  # +1 because diff reduces length by 1
    ax1.axvline(x=trigger_frame_idx + 1, color='black', linestyle='--', alpha=0.8, linewidth=2)
    ax1.text(trigger_frame_idx + 1.2, 0.5, "Anomaly Trigger Detected", rotation=90, verticalalignment='center', fontweight='bold')
    
    # Bottom Plot: Filmstrip of Key Frames
    # Select 4 evenly spaced frames to represent the video visually
    key_frames = np.linspace(0, num_frames - 1, 4, dtype=int)
    
    for i, frame_idx in enumerate(key_frames):
        ax2 = plt.subplot2grid((3, 4), (2, i))
        ax2.imshow(video_array[frame_idx])
        ax2.set_title(f"Frame {frame_idx + 1}\nScore: {temporal_scores[frame_idx]:.2f}", fontsize=10)
        ax2.axis('off')

    plt.tight_layout()
    save_path = os.path.join(XAI_TRAJECTORY_DIR, f"trajectory_rank_{rank+1}_score_{final_score:.2f}.png")
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close(fig) 

print(f"\n✅ Temporal Trajectory Complete! 10 Images saved in '{XAI_TRAJECTORY_DIR}/'.")

Temporal Trajectory outputs will be saved to: thesis_xai_trajectory/
Loading Nano3D Model for Temporal Analysis...

Scanning Test Dataset for the Best Anomaly Examples...
Executing Temporal Unrolling on Top 10 anomaly videos...

Rank  1 | Final Score: 0.9891 | Max Jump: +0.5107
Rank  2 | Final Score: 0.9886 | Max Jump: +0.4658
Rank  3 | Final Score: 0.9826 | Max Jump: +0.2585
Rank  4 | Final Score: 0.9803 | Max Jump: +0.2988
Rank  5 | Final Score: 0.9776 | Max Jump: +0.2301
Rank  6 | Final Score: 0.9759 | Max Jump: +0.1828
Rank  7 | Final Score: 0.9758 | Max Jump: +0.2737
Rank  8 | Final Score: 0.9735 | Max Jump: +0.3779
Rank  9 | Final Score: 0.9730 | Max Jump: +0.3532
Rank 10 | Final Score: 0.9724 | Max Jump: +0.2542

✅ Temporal Trajectory Complete! 10 Images saved in 'thesis_xai_trajectory/'.


In [15]:
!pip install umap-learn seaborn

In [18]:
# ==========================================================
# BLOCK 44 (XAI 5): LATENT SPACE VISUALIZATION (UMAP)
# ==========================================================
import os
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
import umap # pip install umap-learn
import seaborn as sns
import gc
from tensorflow.keras.backend import clear_session

# 1. Setup Output Directory
# ----------------------------------------------------------
XAI_LATENT_DIR = "thesis_xai_latent"
os.makedirs(XAI_LATENT_DIR, exist_ok=True)
print(f"Latent Space outputs will be saved to: {XAI_LATENT_DIR}/")

# 2. Clear Memory & Load Model
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared. Loading Nano3D Model for Latent Extraction...")

try:
    model = load_model('best_nano3d_model.keras', custom_objects={'swish': tf.nn.swish})
except Exception as e:
    print("Error loading model. Ensure Nano3D training is complete.")
    raise e

# 3. Create Feature Extractor
# ----------------------------------------------------------
# Find the dense layer just BEFORE the final sigmoid
target_layer_name = None
for layer in reversed(model.layers[:-1]): 
    if isinstance(layer, tf.keras.layers.Dense):
        target_layer_name = layer.name
        break

print(f"Extracting 128D embeddings from layer: {target_layer_name}")

feature_extractor = tf.keras.models.Model(
    inputs=model.inputs, 
    outputs=model.get_layer(target_layer_name).output
)

# 4. Extract Embeddings for the Entire Test Set (MEMORY SAFE)
# ----------------------------------------------------------
print("Scanning Test Dataset... (This will run much faster now)")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

true_labels = []
all_embeddings = []

total_batches = len(test_generator)
start_time = time.time()

for i in range(total_batches):
    X_batch, y_batch = test_generator[i]
    
    # CRITICAL FIX: Direct tensor call bypasses the .predict() memory leak
    # We convert to tensor, run forward pass, and convert back to numpy
    tensor_batch = tf.convert_to_tensor(X_batch, dtype=tf.float32)
    embeddings = feature_extractor(tensor_batch, training=False).numpy()
    
    all_embeddings.extend(embeddings)
    true_labels.extend(y_batch)
    
    if (i+1) % 10 == 0 or (i+1) == total_batches:
        print(f"Processed {i+1}/{total_batches} batches...")

all_embeddings = np.array(all_embeddings)
true_labels = np.array(true_labels)

print(f"Extracted features shape: {all_embeddings.shape}")
print(f"Extraction took {time.time() - start_time:.2f} seconds.")

# 5. Perform UMAP Dimensionality Reduction
# ----------------------------------------------------------
print("\nRunning UMAP Dimensionality Reduction...")
# UMAP is faster and preserves global structure better than t-SNE
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, metric='cosine', random_state=42)
embeddings_2d = reducer.fit_transform(all_embeddings)
print("UMAP complete.")

# 6. Plot the Latent Space
# ----------------------------------------------------------
print("Generating Scatter Plot...")
plt.figure(figsize=(12, 10), dpi=300)
sns.set_style("whitegrid")

# Separate data by class
normal_mask = (true_labels == 0)
anomaly_mask = (true_labels == 1)

# Plot Normal Videos (Blue)
plt.scatter(
    embeddings_2d[normal_mask, 0], 
    embeddings_2d[normal_mask, 1], 
    c='royalblue', label='Normal (Class 0)', 
    alpha=0.7, s=40, edgecolors='w', linewidths=0.5
)

# Plot Anomaly Videos (Red)
plt.scatter(
    embeddings_2d[anomaly_mask, 0], 
    embeddings_2d[anomaly_mask, 1], 
    c='crimson', label='Anomaly (Class 1)', 
    alpha=0.7, s=40, edgecolors='w', linewidths=0.5
)

plt.title("UMAP Latent Space Visualization of Nano3D Embeddings", fontsize=18, fontweight='bold', pad=15)
plt.xlabel("UMAP Dimension 1", fontsize=14)
plt.ylabel("UMAP Dimension 2", fontsize=14)

# Customize Legend
plt.legend(title="Video Classes", title_fontsize='13', fontsize='12', loc='best', markerscale=1.5)

# Add text box with model stats
textstr = f"Model: Nano3D\nAugmentation: CutMix\nTest Samples: {len(true_labels)}"
props = dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray')
plt.gca().text(0.02, 0.02, textstr, transform=plt.gca().transAxes, fontsize=12,
        verticalalignment='bottom', bbox=props)

plt.tight_layout()

# Save High-Res Image
save_path = os.path.join(XAI_LATENT_DIR, "nano3d_umap_latent_space.png")
plt.savefig(save_path, bbox_inches='tight', dpi=300)
plt.close()

print(f"✅ Latent Space Visualization Complete! Saved to '{save_path}'")

Latent Space outputs will be saved to: thesis_xai_latent/
GPU Memory cleared. Loading Nano3D Model for Latent Extraction...
Extracting 128D embeddings from layer: dense_12
Scanning Test Dataset... (This will run much faster now)
Processed 10/113 batches...
Processed 20/113 batches...
Processed 30/113 batches...
Processed 40/113 batches...
Processed 50/113 batches...
Processed 60/113 batches...
Processed 70/113 batches...
Processed 80/113 batches...
Processed 90/113 batches...
Processed 100/113 batches...
Processed 110/113 batches...
Processed 113/113 batches...
Extracted features shape: (452, 128)
Extraction took 5.61 seconds.

Running UMAP Dimensionality Reduction...
UMAP complete.
Generating Scatter Plot...
✅ Latent Space Visualization Complete! Saved to 'thesis_xai_latent/nano3d_umap_latent_space.png'
